#Disease clasification on plant_disease data


Importing modules
---

In [1]:
pip install numpy pandas matplotlib seaborn scikit-learn pillow torch torchvision timm

Note: you may need to restart the kernel to use updated packages.


In [29]:
import os, re, random, numpy as np
import seaborn as sns
from pathlib import Path
from collections import defaultdict
from collections import Counter
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import average_precision_score
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image
import timm

In [3]:
folders = os.listdir('train')
print(len(folders))
print(folders[:10])

83
['tomato late blight', 'cucumber bacterial wilt', 'rice sheath blight', 'coffee black rot', 'celery anthracnose', 'banana black leaf streak', 'soybean downy mildew', 'peach scab', 'broccoli downy mildew', '.DS_Store']


Data transformation
---

In [4]:
MULTI_WORD_PLANTS = {"bell pepper"}

def parse_folders(root_dir):
    root = Path(root_dir)
    folders = sorted([f for f in root.iterdir() if f.is_dir()])

    disease_set, plant_set = set(), set()
    folder_meta = []

    for folder in folders:
        name = folder.name.lower().strip()
        if name == ".ds_store":
            continue

        matched_plant = next((p for p in MULTI_WORD_PLANTS if name.startswith(p + " ")), None)
        if matched_plant:
            disease_name = name[len(matched_plant)+1:]
        else:
            parts = name.split()
            matched_plant = parts[0]
            disease_name = " ".join(parts[1:])

        disease_set.add(disease_name)
        plant_set.add(matched_plant)
        folder_meta.append((folder, matched_plant, disease_name))

    disease2idx = {d: i for i, d in enumerate(sorted(disease_set))}
    plant2idx = {p: i for i, p in enumerate(sorted(plant_set))}

    print(f"Unique diseases : {len(disease2idx)}")
    print(f"Unique plants   : {len(plant2idx)}")

    samples = []
    for folder, plant_name, disease_name in folder_meta:
        d_idx = disease2idx[disease_name]
        p_idx = plant2idx[plant_name]
        exts = {".jpg", ".jpeg", ".png", ".JPG", ".JPEG", ".PNG"}
        for img_path in folder.iterdir():
            if img_path.suffix in exts:
                samples.append((str(img_path), d_idx, p_idx))

    print(f"Total images: {len(samples)}")
    return samples, disease2idx, plant2idx

In [5]:
TRAIN_TF = transforms.Compose([
    transforms.RandomResizedCrop(224, scale=(0.7, 1.0)),

    transforms.RandomHorizontalFlip(p=0.5),

    transforms.RandomRotation(10),

    transforms.ColorJitter(
        brightness=0.3,
        contrast=0.3,
        saturation=0.25,
        hue=0.05
    ),
    transforms.RandomApply([
        transforms.GaussianBlur(kernel_size=3)
    ], p=0.2),

    transforms.ToTensor(),

    transforms.RandomErasing(p=0.25, scale=(0.02, 0.15)),

    transforms.Normalize([0.485, 0.456, 0.406],
                         [0.229, 0.224, 0.225]),
])

VAL_TF = transforms.Compose([
    transforms.Resize(256),          # keep aspect ratio
    transforms.CenterCrop(224),      # deterministic crop
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406],
                         [0.229, 0.224, 0.225]),
])

In [6]:
class PlantDataset(Dataset):
    def __init__(self, samples, transform):
        self.samples   = samples
        self.transform = transform

    def __len__(self): return len(self.samples)

    def __getitem__(self, idx):
        path, d_idx, p_idx = self.samples[idx]
        img = Image.open(path).convert("RGB")
        img = self.transform(img)
        return img, d_idx, p_idx

creating loaders

In [7]:

train_samples, disease2idx, plant2idx = parse_folders("train")
val_samples, _, _ = parse_folders("val")
full_samples = train_samples + val_samples

test_plant = ["bean", "tobacco", "broccoli"]
idx2plant = {v: k for k, v in plant2idx.items()}

train_plants = []
test_plants  = []

for path, d_idx, p_idx in full_samples:
    plant_name = idx2plant[p_idx]

    if plant_name in test_plant:
        test_plants.append((path, d_idx, p_idx))
    else:
        train_plants.append((path, d_idx, p_idx))

Unique diseases : 39
Unique plants   : 32
Total images: 7783
Unique diseases : 39
Unique plants   : 31
Total images: 390


In [8]:
print(len(disease2idx))
len(plant2idx)

39


32

In [9]:
train_ds = PlantDataset(train_plants, transform=TRAIN_TF)
test_ds  = PlantDataset(test_plants,  transform=VAL_TF)

In [10]:
train_loader = DataLoader(train_ds, batch_size=32, shuffle=True, num_workers=0)
test_loader  = DataLoader(test_ds,  batch_size=32, shuffle=False, num_workers=0)

In [11]:
print("Train size:", len(train_plants))
print("Test size:", len(test_plants))

train_set = set(idx2plant[s[2]] for s in train_plants)
test_set  = set(idx2plant[s[2]] for s in test_plants)

print("Overlap:", train_set & test_set)

Train size: 7805
Test size: 368
Overlap: set()


Model Implementation
---

Gradient reversal layer

In [12]:
class GradientReversalFn(torch.autograd.Function):
    @staticmethod
    def forward(ctx, x, lambda_):
        ctx.save_for_backward(torch.tensor(lambda_))
        return x.clone()

    @staticmethod
    def backward(ctx, grad_output):
        lambda_, = ctx.saved_tensors
        return -lambda_ * grad_output, None   # reverse + scale

class GRL(nn.Module):
    def __init__(self): super().__init__()
    def forward(self, x, lambda_): return GradientReversalFn.apply(x, lambda_)

def get_lambda(epoch, total_epochs):
    p = epoch / total_epochs
    return (2 / (1 + np.exp(-5 * p)) - 1)


Using EfficentNet0 as a base unfreezing last 2 layers projectiong to another space and applying 2 classification heads(disease and plant)

In [13]:
class DiseaseClassifier(nn.Module):
    def __init__(self, num_diseases, num_plants):
        super().__init__()

        self.backbone = timm.create_model(
            "efficientnet_b0", pretrained=True, num_classes=0
        )
        feat_dim = self.backbone.num_features

        for p in self.backbone.parameters():
            p.requires_grad = False

        for block in self.backbone.blocks[-2:]:
            for p in block.parameters():
                p.requires_grad = True

        self.projector = nn.Sequential(
            nn.Linear(feat_dim, 512),
            nn.BatchNorm1d(512),
            nn.ReLU(),
            nn.Dropout(0.3),
        )
        #Disease classification head
        self.disease_head = nn.Sequential(
            nn.Linear(512, 256),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(256, num_diseases),
        )

        #Plant classification head
        self.grl = GRL()
        self.domain_head = nn.Sequential(
            nn.Linear(512, 256),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(256, num_plants),
        )

    def forward(self, x, lambda_=1.0):
        feat = self.backbone(x)
        feat = self.projector(feat)

        #Disease prediction
        disease_logits = self.disease_head(feat)

        #Domain adversarial prediction (gradient reversed)
        feat_rev      = self.grl(feat, lambda_)
        domain_logits = self.domain_head(feat_rev)

        return disease_logits, domain_logits


In [14]:
pip install wandb -q

Note: you may need to restart the kernel to use updated packages.


In [15]:
import wandb
wandb.login()

wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from /Users/annatitizyan/.netrc.
wandb: Currently logged in as: titizyan-anna (titizyan-anna-yerevan-state-university-ysu) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


True

In [16]:
wandb.init(
    project="Plant_Disease_Classification",
    name="experiment-14",
    config={
        "learning_rate": 1e-4,
        "epochs": 30,
        "batch_size": 32,
        "weight_decay": 1e-4
    }
)

In [17]:
print(torch.backends.mps.is_available())

True


In [18]:
device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
print(device)

mps


In [19]:
def compute_map(y_true, y_prob, num_classes):
    y_true = np.array(y_true)
    y_prob = np.array(y_prob)

    y_true_onehot = np.eye(num_classes)[y_true]

    ap_per_class = []

    for c in range(num_classes):
        # skip empty classes
        if np.sum(y_true_onehot[:, c]) == 0:
            continue

        ap = average_precision_score(
            y_true_onehot[:, c],
            y_prob[:, c]
        )
        ap_per_class.append(ap)

    return np.mean(ap_per_class)

Training
---

In [20]:
def train_epoch(model, loader, optimizer, criterion, epoch, total_epochs, device):
    model.train()

    # GRL schedule (keep yours but soften effect)
    lambda_ =  get_lambda(epoch, total_epochs)
    if epoch < 4:
        lambda_ = 0.0

    total_loss = 0

    all_probs = []
    all_labels = []

    disease_correct = 0
    disease_total = 0
    plant_correct = 0
    plant_total = 0

    for imgs, d_labels, p_labels in loader:
        imgs = imgs.to(device)
        d_labels = d_labels.to(device)
        p_labels = p_labels.to(device)

        optimizer.zero_grad()

        disease_logits, plant_logits = model(imgs, lambda_)
        d_loss = criterion(disease_logits, d_labels)
        p_loss = criterion(plant_logits, p_labels)
        loss = d_loss + 0.3 * p_loss

        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        d_preds = disease_logits.argmax(dim=1)
        disease_correct += (d_preds == d_labels).sum().item()

        disease_total += d_labels.size(0)
        p_preds = plant_logits.argmax(dim=1)
        plant_correct += (p_preds == p_labels).sum().item()
        plant_total += p_labels.size(0)

        probs = torch.softmax(disease_logits, dim=1)
        all_probs.append(probs.detach().cpu())
        all_labels.append(d_labels.detach().cpu())

    all_probs = torch.cat(all_probs).numpy()
    all_labels = torch.cat(all_labels).numpy()

    mAP = compute_map(all_labels, all_probs, len(disease2idx))

    avg_loss = total_loss / len(loader)
    acc = 100 * disease_correct / disease_total
    plant_acc = 100 * plant_correct / plant_total

    print(
        f"[Train] λ={lambda_:.3f} | "
        f"loss={avg_loss:.4f} | "
        f"acc_disease={acc:.2f}% | "
        f"acc_plant={plant_acc:.2f}% | "
        f"mAP={mAP:.4f}"
    )

    return avg_loss, mAP

In [21]:
@torch.no_grad()
def val_epoch(model, loader, criterion, device, num_classes):
    model.eval()

    total_loss = 0
    correct = total = 0

    all_probs = []
    all_labels = []

    for imgs, d_labels, _ in loader:
        imgs = imgs.to(device)
        d_labels = d_labels.to(device)

        disease_logits, _ = model(imgs, lambda_=0.0)
        loss = criterion(disease_logits, d_labels)

        total_loss += loss.item()

        probs = torch.softmax(disease_logits, dim=1)

        preds = probs.argmax(dim=1)

        correct += (preds == d_labels).sum().item()
        total += len(d_labels)

        all_probs.append(probs.cpu().numpy())
        all_labels.append(d_labels.cpu().numpy())

    all_probs = np.concatenate(all_probs, axis=0)
    all_labels = np.concatenate(all_labels, axis=0)

    acc = 100 * correct / total
    map_score = compute_map(all_labels, all_probs, num_classes)

    print(f"  [Val] loss={total_loss/len(loader):.4f} | "
          f"acc={acc:.2f}% | mAP={map_score:.4f}")

    return acc, map_score

In [22]:

# ── Model ─────────────────────────────────────────────────
num_diseases = len(disease2idx)
num_plants   = len(plant2idx)

model = DiseaseClassifier(num_diseases, num_plants).to(device)

    # Count trainable params
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total_p   = sum(p.numel() for p in model.parameters())
print(f"\nTrainable params: {trainable:,} / {total_p:,} "
        f"({100*trainable/total_p:.1f}%)\n")

optimizer = torch.optim.AdamW(
        filter(lambda p: p.requires_grad, model.parameters()),
        lr=3e-5, weight_decay=1e-4
)

scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
        optimizer, T_max=30, eta_min=1e-6
)

criterion = nn.CrossEntropyLoss()

    # ── Training ─────────────────────────────────────────────
best_map = 0
epochs = 30

for epoch in range(1, epochs + 1):
    print(f"\nEpoch {epoch}/{epochs}")

    avg_train_loss,train_mAP = train_epoch(model, train_loader, optimizer, criterion,
        epoch, epochs, device)
    test_acc, test_map = val_epoch(model, test_loader, criterion, device, num_diseases)
    wandb.log({
    "train/mAP": train_mAP,
    "val/mAP": test_map,
    "epoch": epoch
})
    scheduler.step()

    if test_map > best_map:
        best_map = test_map
        print(f"✅ best model (mAP={test_map:.4f})")

print(f"\n🏆 Best Val mAP: {best_map:.4f}")
wandb.finish()


Trainable params: 3,681,379 / 4,945,347 (74.4%)


Epoch 1/30
[Train] λ=0.000 | loss=4.4058 | acc_disease=11.81% | acc_plant=14.81% | mAP=0.0538
  [Val] loss=3.4216 | acc=13.86% | mAP=0.2254
✅ best model (mAP=0.2254)

Epoch 2/30
[Train] λ=0.000 | loss=3.8092 | acc_disease=23.96% | acc_plant=22.29% | mAP=0.1543
  [Val] loss=3.2106 | acc=22.83% | mAP=0.3013
✅ best model (mAP=0.3013)

Epoch 3/30
[Train] λ=0.000 | loss=3.3270 | acc_disease=32.61% | acc_plant=33.50% | mAP=0.2511
  [Val] loss=3.1099 | acc=23.10% | mAP=0.3416
✅ best model (mAP=0.3416)

Epoch 4/30
[Train] λ=0.322 | loss=2.9529 | acc_disease=39.78% | acc_plant=39.49% | mAP=0.3383
  [Val] loss=2.9304 | acc=26.90% | mAP=0.4040
✅ best model (mAP=0.4040)

Epoch 5/30
[Train] λ=0.394 | loss=2.6650 | acc_disease=45.56% | acc_plant=43.06% | mAP=0.4108
  [Val] loss=2.8331 | acc=24.18% | mAP=0.4535
✅ best model (mAP=0.4535)

Epoch 6/30
[Train] λ=0.462 | loss=2.3996 | acc_disease=51.29% | acc_plant=46.69% | mAP=0.4727
  [Val] loss=2.6428 

wandb: ERROR The nbformat package was not found. It is required to save notebook history.


  [Val] loss=1.8667 | acc=47.01% | mAP=0.6377

🏆 Best Val mAP: 0.6440


epoch,▁▁▁▂▂▂▂▃▃▃▃▄▄▄▄▅▅▅▅▆▆▆▆▇▇▇▇███
train/mAP,▁▂▃▄▄▅▅▆▆▆▇▇▇▇▇▇██████████████
val/mAP,▁▂▃▄▅▅▆▆▆▇▇▇▇█▇███████████████
epoch,30
train/mAP,0.80209
val/mAP,0.63772


In [23]:
def predict(image_path,disease2idx,plant2idx, checkpoint_path="best_model.pth"):
    ckpt = torch.load(checkpoint_path, map_location=device, weights_only=False)

    idx2disease = {v: k for k, v in disease2idx.items()}

    model = DiseaseClassifier(len(disease2idx), len(plant2idx)).to(device)
    model.load_state_dict(ckpt["model_state_dict"])
    model.eval()

    img = Image.open(image_path).convert("RGB")
    img = VAL_TF(img).unsqueeze(0).to(device)

    with torch.no_grad():
        outputs = model(img)
        logits = outputs[0] if isinstance(outputs, tuple) else outputs
        probs = F.softmax(logits, dim=1)[0]
        top5 = probs.topk(5)

    print(f"\nPredictions for: {image_path}")
    for prob, idx in zip(top5.values, top5.indices):
        print(f"{idx2disease[idx.item()]:35s}  {100*prob.item():.2f}%")

In [24]:
predict('/Users/annatitizyan/Desktop/Plant_disease/val/apple scab/apple_scab_google_0060.jpg',disease2idx,plant2idx)


Predictions for: /Users/annatitizyan/Desktop/Plant_disease/val/apple scab/apple_scab_google_0060.jpg
scab                                 40.45%
black rot                            14.37%
late blight                          11.48%
brown rot                            9.23%
downy mildew                         4.77%
